# LLM Colors Preference Analysis
Analyze and compare results of adaptive preference elicitation (GRUM) from LLMs across models, embeddings, and sampling criteria.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

# Global Configuration
color_names = ["blue", "red", "green", "purple", "yellow"]
color_palette = {"blue": "#1f77b4", "red": "#d62728", "green": "#2ca02c", "purple": "#9467bd", "yellow": "#d6d627"}

## 1. Load Data
We load results from the latest run directory. Each result file contains metadata identifying the model, criterion, and embedding method.

In [2]:
# Set experiment directory
EXP_DIR = ROOT / "results" / "llm" / "llm_colors-20260326-114531"

results_data = []
output_dir = EXP_DIR / "outputs"

if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            res = json.load(j)
            
            # Metadata extraction
            criterion = res.get("criterion", "social").capitalize()
            embedding = res.get("embedding_method", "")
            if not embedding:
                embedding = "HS" if "hidden_state" in f.name else "ST"
            else:
                embedding = "HS" if "hidden_state" in embedding else "ST"
            
            model_full = res.get("model_id", "").split("/")[-1]
            model_type = "Instruct" if "Instruct" in model_full else "Pretrained"
            
            # Data extraction
            history = res.get("history", {})
            if not history: continue
            
            last_step = str(sorted(map(int, history.keys()))[-1])
            delta = history[last_step]["grum"]["delta"]
            
            for i, color in enumerate(color_names):
                results_data.append({
                    "Model Type": model_type,
                    "Model ID": model_full,
                    "Embedding": embedding,
                    "Criterion": criterion,
                    "Color": color,
                    "Score (Delta)": delta[i],
                    # Add combined dimensions for exhaustive grids
                    "Emb-Crit": f"{embedding} | {criterion}",
                    "Mod-Crit": f"{model_type} | {criterion}",
                    "Mod-Emb": f"{model_type} | {embedding}"
                })

df = pd.DataFrame(results_data)
print(f"Loaded {len(df) // len(color_names)} experiment runs.")
display(df.drop_duplicates(['Model ID', 'Embedding', 'Criterion'])[['Model ID', 'Embedding', 'Criterion']])

## 2. Comparative Analysis (Exhaustive Grids)
In the following sections, we compare experimental dimenions side-by-side. Each run is shown in its own subplot to avoid averaging results and to maintain absolute clarity. No error bars are shown, and bar colors correspond to the actual color names.

### 2.1 Model Type Comparison: Pretrained vs Instruct
This grid shows how Pretrained (left) and Instruct (right) models perform across all embedding and elicitation combinations.

In [3]:
g = sns.FacetGrid(df, row="Emb-Crit", col="Model Type", height=3, aspect=2, margin_titles=True, sharey='row')
g.map_dataframe(sns.barplot, x="Color", y="Score (Delta)", palette=color_palette, hue="Color", dodge=False, errorbar=None)
g.add_legend(title="Color Item")
g.set_axis_labels("Color", "Delta")
g.set_titles(row_template="{row_name}", col_template="{col_name} Model")
plt.subplots_adjust(top=0.92)
g.fig.suptitle("Model Type Direct Comparison (Pretrained vs Instruct)", fontsize=16)
plt.show()

### 2.2 Embedding Method Comparison: HS vs ST
Compare Hidden State (left) vs Sentence Transformer (right) embeddings for every model/criterion pair.

In [4]:
g = sns.FacetGrid(df, row="Mod-Crit", col="Embedding", height=3, aspect=2, margin_titles=True, sharey='row')
g.map_dataframe(sns.barplot, x="Color", y="Score (Delta)", palette=color_palette, hue="Color", dodge=False, errorbar=None)
g.add_legend(title="Color Item")
g.set_axis_labels("Color", "Delta")
g.set_titles(row_template="{row_name}", col_template="{col_name} Embedding")
plt.subplots_adjust(top=0.92)
g.fig.suptitle("Embedding Method Direct Comparison (HS vs ST)", fontsize=16)
plt.show()

### 2.3 Elicitation Criterion Comparison: Social vs Personalized vs Random
Compare the three sampling methodologies side-by-side across all model/embedding pairs.

In [5]:
g = sns.FacetGrid(df, row="Mod-Emb", col="Criterion", height=3, aspect=1.8, margin_titles=True, sharey='row')
g.map_dataframe(sns.barplot, x="Color", y="Score (Delta)", palette=color_palette, hue="Color", dodge=False, errorbar=None)
g.add_legend(title="Color Item")
g.set_axis_labels("Color", "Delta")
g.set_titles(row_template="{row_name}", col_template="{col_name} Criterion")
plt.subplots_adjust(top=0.92)
g.fig.suptitle("Elicitation Criterion Direct Comparison", fontsize=16)
plt.show()

## 3. The Identifiability Problem: Unbounded Utility Drift

For completeness, we include the historical drift analysis showing the behavior of the system before sum-to-zero normalization was enforced.

### Visualizing the Historical Drift
Below we load results from the `colors_unbounded` artifact. Bar and line colors correspond to the actual color names.

In [ ]:
from grums.experiments.utils import load_experiment_results

# Path to merged unbounded data in artifacts
artifact_path = "/home/lotanamit/grum4llm/artifacts/llm/colors_unbounded"

unbounded_results = {}
for method in ['hs', 'st']:
    path = os.path.join(artifact_path, method)
    if os.path.exists(path):
        unbounded_results[method] = load_experiment_results(path)

if unbounded_results:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    for i, (method, res) in enumerate(unbounded_results.items()):
        social_runs = [k for k in res.keys() if 'social' in k]
        if not social_runs: continue
        
        sample_key = social_runs[0]
        history = res[sample_key]['history']
        
        steps = sorted(map(int, history.keys()))
        deltas = [history[str(s)]['grum']['delta'] for s in steps]
        deltas = np.array(deltas)
        
        for j in range(deltas.shape[1]):
            color_name = color_names[j]
            axes[i].plot(steps, deltas[:, j], label=color_name, color=color_palette[color_name])
        
        axes[i].set_title(f'Unbounded Utility Drift ({method.upper()})')
        axes[i].set_xlabel('Elicitation Step')
        axes[i].set_ylabel('Utility (Delta)')
        axes[i].legend(ncol=2)
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()